# Trabajo Práctico N°1: Redes Neuronales Básicas con Keras

## Dependencias

Si estás trabajando con Google Colab, no son necesarias dependencias adicionales, puesto que Keras, Pytorch y Matplotlib vienen por defecto. Si deseas ejecutar este notebook en local, necesitarás instalar las siguientes dependencias:

In [ ]:
!pip install torch keras torchvision matplotlib
# o
#!uv pip install torch keras torchvision matplotlib

## Backend

Utilizaremos la biblioteca `pytorch` como backend de Keras. Para ello, debemos ejecutar la siguiente celda antes de cualquier importación que incluya `keras`:

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

## Dataset

El dataset **Fashion MNIST** incluye imágenes de 28x28 píxeles en escala de grises de 10 tipos de prendas:

| Etiqueta numérica | Descripción |
|-------|---------------|
| 0 | Camiseta |
| 1 | Pantalón |
| 2 | Pulóver |
| 3 | Vestido |
| 4 | Abrigo |
| 5 | Sandalia |
| 6 | Camisa |
| 7 | Zapatilla de deporte |
| 8 | Bolso |
| 9 | Botín |

In [ ]:
from keras.datasets.fashion_mnist import load_data

(X_train, y_train), (X_test, y_test) = load_data()

# Nombres de las clases, en el mismo orden que las etiquetas numéricas
class_names = ["Camiseta", "Pantalón", "Pulóver", "Vestido", "Abrigo",
               "Sandalia", "Camisa", "Zapatilla de deporte", "Bolso", "Botín"]

## Actividades

### Exploración del dataset

Verificamos el resultado de `X_train.shape` y `X_test.shape`.

In [ ]:
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

1. ¿Con cuántas imágenes cuenta el **set de entrenamiento** del dataset? ¿Y el **set de test**?

**Respuesta**: El set de entrenamiento tiene **60.000 imágenes** (`X_train.shape = (60000, 28, 28)`) y el set de test tiene **10.000 imágenes** (`X_test.shape = (10000, 28, 28)`). La primera dimensión del tensor es la cantidad de ejemplos.

2. ¿Qué tamaño (ancho por alto) tiene cada imagen del dataset?

**Respuesta**: Cada imagen mide **28 x 28 píxeles** (las dos últimas dimensiones del `shape`). Cada píxel es un entero entre 0 y 255 que representa la intensidad de gris.

3. ¿Cómo creen que `shape` cambiaría si se tratase de imágenes a color, en lugar de escalas de grises?

**Respuesta**: Se agregaría una dimensión más para los **canales de color**. Con imágenes RGB el shape sería `(60000, 28, 28, 3)`: para cada píxel habría tres valores (rojo, verde y azul) en lugar de uno solo. En escala de grises el canal es único, por eso Keras lo omite (equivaldría a `(60000, 28, 28, 1)`).

4. Muestren al menos diez imágenes del dataset usando `matplotlib`.

Para obtener una sola imagen indexamos el tensor por su primera dimensión: `X_train[0]` devuelve un tensor 2D de 28x28.

In [ ]:
import matplotlib.pyplot as plt

first_img = X_train[0]  # indexar la primera dimensión devuelve una sola imagen 2D
plt.imshow(first_img, cmap='gray')

# Desactivar ejes del gráfico
plt.axis('off')
plt.title(class_names[y_train[0]])
plt.show()

In [ ]:
# Grilla con las primeras 10 imágenes y su etiqueta
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"{y_train[i]}: {class_names[y_train[i]]}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Preprocesamiento

5. Normalizamos los píxeles para que queden en el rango [0, 1]. Las redes neuronales entrenan de forma más estable cuando las entradas tienen valores pequeños y en una escala similar.

In [ ]:
print("Antes:", X_train.min(), X_train.max(), X_train.dtype)

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

print("Después:", X_train.min(), X_train.max(), X_train.dtype)

### Configuración de la red

6. Configuramos la red neuronal:
  - a) Capa de entrada con la forma de cada imagen: `(28, 28)`.
  - b) Capa `Flatten` para aplanar cada imagen en un vector de 784 valores.
  - c) Dos capas densas ocultas (128 y 64 neuronas, activación ReLU).
  - d) Como es una **clasificación multiclase**, la capa de salida necesita la activación **softmax**, con 10 neuronas (una por clase). Softmax convierte las salidas en probabilidades que suman 1.

In [ ]:
import keras
from keras import layers

model = keras.Sequential([
    keras.Input(shape=(28, 28)),             # a) forma de entrada
    layers.Flatten(),                         # b) aplanar 28x28 -> 784
    layers.Dense(128, activation="relu"),     # c) capa densa 1
    layers.Dense(64, activation="relu"),      # c) capa densa 2
    layers.Dense(10, activation="softmax"),   # d) salida multiclase
])

model.summary()

### Entrenamiento

7. Compilamos la red con:
   - a) El optimizador **Adam**.
   - b) La función de pérdida **Sparse Categorical Cross Entropy** (las etiquetas son enteros 0-9, no vectores one-hot).
   - c) La métrica **Accuracy**.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

8. Entrenamos el modelo durante 10 épocas, reservando un 10% de los datos de entrenamiento para validación.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
)

### Evaluación

9. Evaluamos el modelo sobre el conjunto de test con `model.evaluate`.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Pérdida en test:  {test_loss:.4f}")
print(f"Accuracy en test: {test_acc:.4f}")

**Respuesta**: Con esta arquitectura el accuracy en test suele quedar alrededor de **87-89%**, lo cual es satisfactorio para una red densa simple (adivinar al azar daría 10%). Igualmente hay margen de mejora: las clases que más se confunden son las parecidas entre sí (camiseta, camisa, pulóver y abrigo). Además, si el accuracy de entrenamiento es bastante mayor que el de test, hay algo de sobreajuste. Una red convolucional (CNN) aprovecharía mejor la estructura espacial de las imágenes y superaría el 90%.

*(Revisar el valor exacto obtenido al ejecutar la celda anterior.)*

10. Mostramos diez imágenes del test con la **etiqueta predicha** y el **porcentaje de confianza** (la probabilidad máxima que asigna softmax). En verde los aciertos y en rojo los errores.

In [ ]:
import numpy as np

probs = model.predict(X_test[:10], verbose=0)   # (10, 10) probabilidades
preds = np.argmax(probs, axis=1)
confs = np.max(probs, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i], cmap='gray')
    color = "green" if preds[i] == y_test[i] else "red"
    ax.set_title(f"Pred: {class_names[preds[i]]}\n{confs[i]*100:.1f}% (real: {class_names[y_test[i]]})",
                 fontsize=8, color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

11. **Learning rate personalizado**: en lugar de pasar el string `"adam"`, se instancia `keras.optimizers.Adam(learning_rate=...)`. El valor por defecto es `0.001`. Probamos con un modelo nuevo (misma arquitectura, pesos desde cero) para comparar en igualdad de condiciones.

In [ ]:
def build_model():
    return keras.Sequential([
        keras.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])

model_lr = build_model()
model_lr.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_lr = model_lr.fit(X_train, y_train, epochs=10, batch_size=32,
                          validation_split=0.1, verbose=1)

lr_loss, lr_acc = model_lr.evaluate(X_test, y_test, verbose=0)
print(f"Adam lr=0.001  -> accuracy test: {test_acc:.4f}")
print(f"Adam lr=0.0005 -> accuracy test: {lr_acc:.4f}")
print("Mejoró" if lr_acc > test_acc else "Empeoró o se mantuvo")

**Respuesta**: El learning rate controla el tamaño del paso que da el optimizador al actualizar los pesos. Un valor más chico (0.0005) hace el entrenamiento más estable pero más lento para converger; uno más grande (por ejemplo 0.01) avanza rápido pero puede oscilar y quedar en una pérdida peor. En esta prueba la diferencia suele ser pequeña (menos de 1 punto de accuracy); el resultado concreto se ve en la salida de la celda anterior. Si se prueba un valor alto como 0.01, normalmente el rendimiento empeora.

## Ejercicio extra (opcional)

12. Graficamos el proceso de entrenamiento: épocas en el eje X, pérdida y accuracy en el eje Y. Lo hacemos de dos formas: con Matplotlib a partir del objeto `History`, y con **TensorBoard** mediante su callback.

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history.history["loss"], marker="o", label="Pérdida (train)")
plt.plot(epochs, history.history["accuracy"], marker="o", label="Accuracy (train)")
plt.plot(epochs, history.history["val_loss"], linestyle="--", label="Pérdida (val)")
plt.plot(epochs, history.history["val_accuracy"], linestyle="--", label="Accuracy (val)")
plt.xlabel("Época")
plt.ylabel("Valor")
plt.title("Proceso de entrenamiento")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Versión con TensorBoard
# (en local: pip install tensorboard)
import datetime

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tb_callback = keras.callbacks.TensorBoard(log_dir=log_dir)

model_tb = build_model()
model_tb.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_tb.fit(X_train, y_train, epochs=10, validation_split=0.1,
             callbacks=[tb_callback], verbose=0)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

Como se observa en el gráfico, la pérdida disminuye rápidamente en las primeras épocas y luego llega a un "piso" donde apenas decrece, mientras que el accuracy sube y se estabiliza. Si la pérdida de validación deja de bajar (o empieza a subir) mientras la de entrenamiento sigue bajando, es señal de sobreajuste.